# Scenario Interpolation Demonstration

## What is Scenario Interpolation?
Scenario interpolation allows you to create a new scenario for an intermediate year by linearly interpolating input values between:
- A start scenario (e.g., 2030)
- An end scenario (e.g., 2050)

For example, if an input value is 50 in 2030 and 100 in 2050, interpolating to 2040 would give you a value of 75.

In [ ]:
# Check the environment is properly configured
from example_helpers import setup_notebook
setup_notebook()

In [ ]:
import pandas as pd
from pyetm.models.session import Session
from pyetm.models.scenario import Scenario

## Example 1: Single Session Interpolation

Interpolation from a single scenario interpolates between the scenario's start year and end year.

In [ ]:
# Create a base scenario for 2050 with some modified inputs
scenario_2050 = Session.new(
    area_code="nl2019",
    end_year=2050,
    title="Netherlands 2050 Base Scenario"
)

scenario_2050.update_user_values({
    "capacity_of_energy_hydrogen_wind_turbine_offshore": 10000.0,
    "capacity_of_energy_power_wind_turbine_offshore": 25000.0,
    "costs_heat_network_storage_mt_steam_hot_water": 6.5,
})

print(f"Created scenario {scenario_2050.id} for {scenario_2050.end_year}")
print(f"Area: {scenario_2050.area_code}")
print(f"Start year: {scenario_2050.start_year}")

In [ ]:
# Interpolate to 2040
scenario_2040 = scenario_2050.interpolate(end_year=2040)

print(f"Created interpolated scenario {scenario_2040.id} for {scenario_2040.end_year}")
print(f"Interpolated from {scenario_2050.start_year} to {scenario_2040.end_year}")

## Example 2: Two-Session Interpolation

Now let's demonstrate interpolation between two distinct sessions.

In [ ]:
# Create a 2030 scenario (conservative targets)
scenario_2030 = Session.new(
    area_code="nl",
    end_year=2030,
    title="Netherlands 2030 Conservative"
)

scenario_2030.update_user_values({
    "capacity_of_energy_hydrogen_wind_turbine_offshore": 10.0,
    "capacity_of_energy_power_wind_turbine_offshore": 25.0,
    "costs_heat_network_storage_mt_steam_hot_water": 6.5,
})

# Create a 2050 scenario (ambitious targets)
scenario_2050_ambitious = Session.new(
    area_code="nl",
    end_year=2050,
    title="Netherlands 2050 Ambitious"
)

scenario_2050_ambitious.update_user_values({
    "capacity_of_energy_hydrogen_wind_turbine_offshore": 1000.0,
    "capacity_of_energy_power_wind_turbine_offshore": 2500.0,
    "costs_heat_network_storage_mt_steam_hot_water": 160.5,
})

print(f"Created 2030 scenario: {scenario_2030.id}")
print(f"Created 2050 scenario: {scenario_2050_ambitious.id}")

In [ ]:
# Interpolate to 2041 using both scenarios
scenario_2040_interp = scenario_2050_ambitious.interpolate(
    end_year=2041,
    start_session=scenario_2030
)

print(f"Created interpolated scenario {scenario_2040_interp.id} for {scenario_2040_interp.end_year}")
print(f"Interpolated between {scenario_2030.end_year} and {scenario_2050_ambitious.end_year}")

## Comparing Input Values to demonstrate effects

In [ ]:
# Get user values for all three scenarios
inputs_2030 = scenario_2030.user_values()
inputs_2040 = scenario_2040_interp.user_values()
inputs_2050 = scenario_2050_ambitious.user_values()

# Create a comparison DataFrame
comparison_data = {
    "2030 (Start)": [],
    "2040 (Interpolated)": [],
    "2050 (End)": [],
    "Change per Year": []
}

keys_to_compare = [
    "capacity_of_energy_hydrogen_wind_turbine_offshore",
    "capacity_of_energy_power_wind_turbine_offshore",
    "costs_heat_network_storage_mt_steam_hot_water"
]

for key in keys_to_compare:
    val_2030 = inputs_2030.get(key, 0)
    val_2040 = inputs_2040.get(key, 0)
    val_2050 = inputs_2050.get(key, 0)

    comparison_data["2030 (Start)"].append(val_2030)
    comparison_data["2040 (Interpolated)"].append(val_2040)
    comparison_data["2050 (End)"].append(val_2050)
    comparison_data["Change per Year"].append((val_2050 - val_2030) / 20)

comparison_df = pd.DataFrame(comparison_data, index=[
    "capacity_of_energy_hydrogen_wind_turbine_offshore",
    "capacity_of_energy_power_wind_turbine_offshore",
    "costs_heat_network_storage_mt_steam_hot_water"
])

print("\nInput Value Comparison:")
print(comparison_df.round(2))

## Example 3: Working with Saved Scenarios

You can also interpolate Saved Scenarios (persisted in MyETM). The interpolated scenario is automatically saved and returned as a SavedScenario.

In [ ]:
# Save the 2030 and 2050 scenarios to MyETM
saved_2030 = scenario_2030.save(title="Demo Conservative 2030")
saved_2050 = scenario_2050_ambitious.save(title="Demo Ambitious 2050")

print(f"Saved scenario {saved_2030.id}: {saved_2030.title}")
print(f"Saved scenario {saved_2050.id}: {saved_2050.title}")

saved_2040 = saved_2050.interpolate(
    end_year=2040,
    start_scenario=saved_2030,
    title="Demo Interpolated 2040",
)

print(f"\nCreated and saved interpolated scenario {saved_2040.id}: {saved_2040.title}")

## Example 4: Batch Interpolation

When you need to interpolate multiple intermediate years at once, use **batch interpolation**. This is more efficient than calling `interpolate()` multiple times, as it makes a single API call to create all interpolated scenarios.

In [ ]:
# Create three scenarios spanning different decades
scenario_2030_batch = Session.new(
    area_code="nl",
    end_year=2030,
    title="Netherlands 2030 Baseline"
)

scenario_2030_batch.update_user_values({
    "capacity_of_energy_hydrogen_wind_turbine_offshore": 50.0,
    "capacity_of_energy_power_wind_turbine_offshore": 200.0,
    "costs_heat_network_storage_mt_steam_hot_water": 10.0,
})

scenario_2050_batch = Session.new(
    area_code="nl",
    end_year=2050,
    title="Netherlands 2050 Medium Growth"
)

scenario_2050_batch.update_user_values({
    "capacity_of_energy_hydrogen_wind_turbine_offshore": 500.0,
    "capacity_of_energy_power_wind_turbine_offshore": 2000.0,
    "costs_heat_network_storage_mt_steam_hot_water": 100.0,
})

scenario_2070_batch = Session.new(
    area_code="nl",
    end_year=2070,
    title="Netherlands 2070 High Growth"
)

scenario_2070_batch.update_user_values({
    "capacity_of_energy_hydrogen_wind_turbine_offshore": 1000.0,
    "capacity_of_energy_power_wind_turbine_offshore": 4000.0,
    "costs_heat_network_storage_mt_steam_hot_water": 200.0,
})

print(f"Created scenario {scenario_2030_batch.id} for 2030")
print(f"Created scenario {scenario_2050_batch.id} for 2050")
print(f"Created scenario {scenario_2070_batch.id} for 2070")

In [ ]:
# Batch interpolate to create 2040 and 2060 scenarios in one call
interpolated_batch = Session.batch_interpolate(
    sessions=[scenario_2030_batch, scenario_2050_batch, scenario_2070_batch],
    end_years=[2040, 2060]
)

print(f"\nCreated {len(interpolated_batch)} interpolated scenarios:")
for scenario in interpolated_batch:
    print(f"  - Scenario {scenario.id} for year {scenario.end_year}")

## Example 5: Batch Interpolation with Saved Scenarios

You can also batch interpolate SavedScenarios. This automatically saves all interpolated scenarios to MyETM.

In [ ]:
# Save the three base scenarios to MyETM
saved_2030_batch = scenario_2030_batch.save(title="Batch Demo 2030")
saved_2050_batch = scenario_2050_batch.save(title="Batch Demo 2050")
saved_2070_batch = scenario_2070_batch.save(title="Batch Demo 2070")

print("Saved base scenarios to MyETM:")
print(f"  - {saved_2030_batch.id}: {saved_2030_batch.title}")
print(f"  - {saved_2050_batch.id}: {saved_2050_batch.title}")
print(f"  - {saved_2070_batch.id}: {saved_2070_batch.title}")

# Batch interpolate and save with custom titles
interpolated_saved = Scenario.batch_interpolate(
    saved_scenarios=[saved_2030_batch, saved_2050_batch, saved_2070_batch],
    end_years=[2040, 2060],
    titles=["Batch Demo 2040 (Interpolated)", "Batch Demo 2060 (Interpolated)"],
    private=False
)

print(f"\nCreated and saved {len(interpolated_saved)} interpolated scenarios:")
for saved in interpolated_saved:
    print(f"  - {saved.id}: {saved.title}")

## Example 6: Minimal Batch

Some attributes are optional

In [ ]:
minimal_interpolated_saved = Scenario.batch_interpolate(
    saved_scenarios=[saved_2030_batch, saved_2050_batch, saved_2070_batch],
    end_years=[2025, 2040, 2060]
)

print(f"\nCreated and saved {len(minimal_interpolated_saved)} interpolated scenarios:")
for saved in minimal_interpolated_saved:
    print(f"  - {saved.id}: {saved.title}")

Using featured scenarios:


In [ ]:
featured = [Scenario.load(5135),Scenario.load(5136)]

featured_interpolated = Scenario.batch_interpolate(
    saved_scenarios = featured,
    end_years = [2025, 2030, 2040, 2045]
)

print(f"\nCreated and saved {len(featured_interpolated)} interpolated scenarios:")
for feat in featured_interpolated:
    print(f"  - {feat.id}: {feat.title}")